In [ ]:

from typing import Sequence, Callable
import warnings

# pylint: disable=protected-access
from functools import reduce, partial
import scipy

import pennylane as qml

from pennylane.transforms.op_transforms import OperationTransformError

from pennylane import transform

from pennylane.typing import TensorLike

import json
import pennylane as qml
import pennylane.numpy as np

class AbsMagnetization(qml.measurements.StateMeasurement):
    """A measurement class that estimates <|M|>."""

    def process_state(self, state, wire_order):
        """Calculates <|M|>.

        Args:
            state (Sequence[complex]): quantum state with a flat shape. It may also have an
                optional batch dimension.

            wire_order (Wires): wires determining the subspace that the state acts on; a matrix of
                dimension 2**n that acts on a subspace of n wires
        
        Returns: 
            abs_mag (float): <|M|>

        
        See the docs for more information:
        https://docs.pennylane.ai/en/stable/code/api/pennylane.measurements.StateMeasurement.html
        """

        state = qml.state().process_state(state, wire_order)

        num_qubits = len(wire_order)
        abs_mag_sum = 0.0

        for i in range(2**num_qubits):
            prob = np.abs(state[i])**2
            
            # Convert index i to its binary representation
            # The bits correspond to the computational basis state |sigma>
            # For a state |s_{N-1}...s_1s_0>, the sum is sum_{j=0}^{N-1} (-1)^{s_j}
            # This is equivalent to (count of 0s) - (count of 1s)
            binary_repr = bin(i)[2:].zfill(num_qubits)
            
            sum_Z_eigenvalues = binary_repr.count('0') - binary_repr.count('1')
            
            abs_mag_sum += prob * np.abs(sum_Z_eigenvalues)
        
        # The run function divides the result by num_qubits,
        # so this class should return N * <|M|>
        return abs_mag_sum


def tfim_ground_state(num_qubits, h):
    """Calculates the ground state of the 1D TFIM Hamiltonian.

    Args:
        num_qubits (int): The number of qubits / spins.
        h (float): The transverse field strength.

    Returns:
        (numpy.tensor): The ground state.
    """

    coeffs = []
    ops = []

    # Nearest-neighbour Z Z terms: -sum_{i=0}^{N-2} Z_i Z_{i+1}
    for i in range(num_qubits - 1):
        coeffs.append(-1.0)
        ops.append(qml.PauliZ(i) @ qml.PauliZ(i + 1))

    # Transverse field X terms: -h sum_{i=0}^{N-1} X_i
    for i in range(num_qubits):
        coeffs.append(-h)
        ops.append(qml.PauliX(i))

    hamiltonian = qml.Hamiltonian(coeffs, ops)
    
    # Get the dense matrix representation of the Hamiltonian
    # Ensure the wire_order is explicitly set for clarity and correctness
    H_matrix = qml.matrix(hamiltonian, wire_order=list(range(num_qubits)))
    
    # Calculate eigenvalues and eigenvectors
    eigenvalues, eigenvectors = qml.math.linalg.eigh(H_matrix)
    
    # The ground state corresponds to the smallest eigenvalue
    ground_state = eigenvectors[:, np.argmin(eigenvalues)]
    
    return ground_state


dev = qml.device("default.qubit")


@qml.qnode(dev)
def magnetization(num_qubits, h):
    """Calculates the absolute value of the magnetization of the 1D TFIM
    Hamiltonian.

    Args:
        num_qubits (int): The number of qubits / spins.
        h (float): The transverse field strength.

    Returns:
        (float): <|M|>.
    """

    # Prepare the ground state of the TFIM Hamiltonian
    ground_state_vec = tfim_ground_state(num_qubits, h)
    qml.StatePrep(ground_state_vec, wires=range(num_qubits))
    
    # Return the custom absolute magnetization measurement
    return AbsMagnetization(wires=list(range(num_qubits)))


def critical_point_estimate(mags, h_values):
    """Provides a finite-size estimate of the critical point of the 1D TFIM
    Hamiltonian. The estimate is done by taking the average value of h for which
    adjacent values of <|M|> differ the most.

    Args:
        mags (numpy.tensor):
            <|M|> values for various values of h (the transverse field strength).
        h_values (numpy.tensor): The transverse field strength values.

    Returns:
        (float): The critical point estimate, h_c.
    """
    differences = [np.abs(mags[i] - mags[i + 1]) for i in range(len(mags) - 1)]
    ind = np.argmax(np.array(differences))

    h_c = np.mean([h_values[ind], h_values[ind + 1]])
    return h_c


def eigvals(op: qml.operation.Operator, k=1, which="SA") -> TensorLike:
    r"""The eigenvalues of one or more operations.

    .. note::

    For a :class:`~.SparseHamiltonian` object, the eigenvalues are computed with the efficient ``scipy.sparse.linalg.eigsh`` method which returns :math:`k` eigenvalues. The default value of :math:`k` is :math:`1`. For an :math:`N \times N` sparse matrix, :math:`k` must be smaller than :math:`N - 1`, otherwise ``scipy.sparse.linalg.eigsh`` fails. If the requested :math:`k` is equal or larger than :math:`N - 1`, the regular ``qml.math.linalg.eigvalsh`` is applied on the dense matrix. For more details see the ``scipy.sparse.linalg.eigsh`` `documentation <https://docs.scipy.org/doc/scipy/reference/generated/scipy.sparse.linalg.eigsh.html#scipy.sparse.linalg.eigsh>`_.

    A second-quantized :mod:`molecular Hamiltonian <pennylane.qchem.molecular_hamiltonian>` is independent of the number of electrons and its eigenspectrum contains the energies of the neutral and charged molecules. Therefore, the `smallest` eigenvalue returned by ``qml.eigvals`` for a molecular Hamiltonian might not always correspond to the neutral molecule.

    Args:
        op (Operator or QNode or QuantumTape or Callable): A quantum operator or quantum circuit.
        k (int): The number of eigenvalues to be returned for a :class:`~.SparseHamiltonian`.
        which (str): Method for computing the eigenvalues of a :class:`~.SparseHamiltonian`. The possible methods are ``'LM'`` (largest in magnitude), ``'SM'`` (smallest in magnitude), ``'LA'`` (largest algebraic), ``'SA'`` (smallest algebraic) and ``'BE'`` (:math:`k/2` from each end of the spectrum).

    Returns:
        TensorLike or qnode (QNode) or quantum function (Callable) or tuple[List[QuantumTape], function]:
        If an operator is provided as input, the eigenvalues are returned directly in the form of a tensor. Otherwise, the transformed circuit is returned as described in :func:`qml.transform <pennylane.transform>`. Executing this circuit will provide the eigenvalues as a tensor.

    Example**

    Given an operation, ``qml.eigvals`` returns the eigenvalues:

    >>> op = qml.Z(0) @ qml.X(1) - 0.5 * qml.Y(1)
    >>> qml.eigvals(op)
    array([-1.11803399, -1.11803399,  1.11803399,  1.11803399])

    It can also be used in a functional form:

    >>> x = torch.tensor(0.6, requires_grad=True)
    >>> eigval_fn = qml.eigvals(qml.RX)
    >>> eigval_fn(x, wires=0)
    tensor([0.9553+0.2955j, 0.9553-0.2955j], grad_fn=<LinalgEigBackward>)

    In its functional form, it is fully differentiable with respect to gate arguments:

    >>> loss = torch.real(torch.sum(eigval_fn(x, wires=0)))
    >>> loss.backward()
    >>> x.grad
    tensor(-0.2955)

    This operator transform can also be applied to QNodes, tapes, and quantum functions that contain multiple operations; see Usage Details below for more details.

    .. details::

    :title: Usage Details

    ``qml.eigvals`` can also be used with QNodes, tapes, or quantum functions that contain multiple operations. However, in this situation, **eigenvalues may be computed numerically**. This can lead to a large computational overhead for a large number of wires.

    Consider the following quantum function:

    .. code-block:: python3

        def circuit(theta):
            qml.RX(theta, wires=1)
            qml.Z(0)

    We can use ``qml.eigvals`` to generate a new function that returns the eigenvalues corresponding to the function ``circuit``:

    >>> eigvals_fn = qml.eigvals(circuit)
    >>> theta = np.pi / 4
    >>> eigvals_fn(theta)
    array([ 0.92387953+0.38268343j,  0.92387953-0.38268343j, -0.92387953+0.38268343j, -0.92387953-0.38268343j])
    """
    if not isinstance(op, qml.operation.Operator):
        if not isinstance(op, (qml.tape.QuantumScript, qml.QNode)) and not callable(op):
            raise OperationTransformError(
                "Input is not an Operator, tape, QNode, or quantum function"
            )
        return _eigvals_tranform(op, k=k, which=which)

    if isinstance(op, qml.Hamiltonian):
        warnings.warn(
            "For Hamiltonians, the eigenvalues will be computed numerically. "
            "This may be computationally intensive for a large number of wires. "
            "Consider using a sparse representation of the Hamiltonian with qml.SparseHamiltonian.",
            UserWarning,
        )
        return qml.math.linalg.eigvalsh(qml.matrix(op))

    if isinstance(op, qml.SparseHamiltonian):
        sparse_matrix = op.sparse_matrix()
        if k < sparse_matrix.shape[0] - 1:
            return scipy.sparse.linalg.eigsh(sparse_matrix, k=k, which=which)[0]
        return qml.math.linalg.eigvalsh(sparse_matrix.toarray())

    # TODO: make `eigvals` take a `wire_order` argument to mimic `matrix`
    try:
        return op.eigvals()
    except qml.operation.EigvalsUndefinedError:
        return eigvals(op.expand(), k=k, which=which)


@partial(transform, is_informative=True)
def _eigvals_tranform(
    tape: qml.tape.QuantumTape, k=1, which="SA"
) -> (Sequence[qml.tape.QuantumTape], Callable):
    def processing_fn(res):
        [qs] = res
        op_wires = [op.wires for op in qs.operations]
        all_wires = qml.wires.Wires.all_wires(op_wires).tolist()
        unique_wires = qml.wires.Wires.unique_wires(op_wires).tolist()

        if len(all_wires) != len(unique_wires):
            warnings.warn(
                "For multiple operations, the eigenvalues will be computed numerically. "
                "This may be computationally intensive for a large number of wires.",
                UserWarning,
            )
        matrix = qml.matrix(qs, wire_order=qs.wires)
        return qml.math.linalg.eigvals(matrix)

    # TODO: take into account wire ordering, by reordering eigenvalues
    # as per operator wires/wire ordering, and by inserting implicit identity
    # matrices (eigenvalues [1, 1]) at missing locations.

    ev = [eigvals(op, k=k, which=which) for op in qs.operations]

    if len(ev) == 1:
        return ev[0]

    return reduce(qml.math.kron, ev)

    return [tape], processing_fn


# These functions are responsible for testing the solution.
def run(test_case_input: str) -> str:
    num_qubits = json.loads(test_case_input)
    h_values = np.arange(0.2, 1.1, 0.005)
    mags = []

    for h in h_values:
        mags.append(magnetization(num_qubits, h) / num_qubits)

    output = critical_point_estimate(np.array(mags), h_values)

    return str(output)


def check(solution_output: str, expected_output: str) -> None:
    solution_output = json.loads(solution_output)
    expected_output = json.loads(expected_output)

    assert np.isclose(solution_output, expected_output, rtol=5e-3)

